# Script pour scrapper les images d'un document Gallica grâce à l'API IIIF

Disponible également sur Google Colab : https://colab.research.google.com/drive/1D0qDo7ZxZDAlr6EVRvjY4AjtHftLfalL?usp=sharing

Ce script récupère toutes les images d'un document à partir de l'API IIIF de Gallica en récupérant le manifeste et en téléchargeant toutes les images progressivement.

Pour lancer le script, appuyez sur `▶️ Tout exécuter` dans le menu en haut.
Si un message d'erreur apparaît, n'hésitez pas à relancer le script de nouveau un peu plus tard, les serveurs de la BnF sont peut-être surchargés.

In [ ]:
identifier = input("Entrer l'identifiant Gallica puis appuyer sur entrée (ex: btv1b84901123) : ")
nom_dossier = input("Entrer le nom du dossier pour stocker les images puis appuyer sur entrée : ")

In [ ]:
drive = False # à changer pour stocker les fichiers sur son Google Drive ; False par défaut (télécharge sur en local à la fin du processus)

In [ ]:
# Délais et retries
delay = 5 # temps de latence entre les téléchargement
modif_delay = 3 # coefficient de modification de temps
max_retries = 3 # nombre maximum d'essai de téléchargement pour une image

quality = "full" # laisser "full" par défaut ou changer par "1000," pour une qualité inférieure

## Fonctions à ne pas modifier

In [ ]:
import os
import requests
from tqdm import tqdm
import zipfile
import time
import sys
from pathlib import Path
import json

if drive == True:
    from google.colab import files

try:
    from google.colab import files
    in_colab = True
except ImportError:
    in_colab = False

In [ ]:
if drive == True:
    from google.colab import drive
    drive.mount('/content/drive')

    base_path = f'/content/drive/MyDrive/gallica_images/{nom_dossier}'
    os.makedirs(base_path, exist_ok=True)
else:
    base_path = os.path.join(os.getcwd(), nom_dossier)
    os.makedirs(base_path, exist_ok=True)

In [ ]:
output_folder = os.path.join(base_path, "images")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
def extract_jpg_urls(identifier, manifest_data=None):
    """
    Extrait les URLs jpg depuis un manifeste IIIF.
    Ordre de priorité :
      1. manifest_data passé en argument
      2. Téléchargement depuis Gallica
      3. Fallback : upload manuel du fichier JSON via Colab
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    url = f"https://gallica.bnf.fr/iiif/ark:/12148/{identifier}/manifest.json"

    data = None

    # 1. Manifest passé directement
    if manifest_data is not None:
        data = json.loads(manifest_data) if isinstance(manifest_data, str) else manifest_data
        print("✅ Manifeste fourni en argument.")

    # 2. Tentative réseau
    if data is None:
        try:
            response = requests.get(url, headers=headers, timeout=(5, 30))
            response.raise_for_status()
            data = response.json()
            print("✅ Manifeste téléchargé depuis Gallica.")
        except Exception as e:
            print(f"⚠️  Réseau indisponible ({type(e).__name__}).")

    # 3. Fallback : upload via Colab ou chemin local
    if data is None:
        print(f"📂 Télécharge le manifeste manuellement depuis :")
        print(f"   {url}")

        if in_colab:
            print("Puis uploade le fichier JSON ci-dessous ↓")
            uploaded = files.upload()

            if not uploaded:
                raise ValueError("❌ Aucun fichier uploadé.")

            filename = next(iter(uploaded))
            content = uploaded[filename]
            data = json.loads(content)
            print(f"✅ Manifeste chargé depuis '{filename}'.")

        else:
            # VSCode / local : saisie du chemin
            manifest_path = input("📁 Chemin vers le fichier JSON du manifeste : ").strip().strip('"').strip("'")

            if not manifest_path or not os.path.exists(manifest_path):
                raise ValueError(f"❌ Fichier introuvable : '{manifest_path}'")

            with open(manifest_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            print(f"✅ Manifeste chargé depuis '{manifest_path}'.")


    # Extraction des URLs
    jpg_urls = []
    for sequence in data.get("sequences", []):
        for canvas in sequence.get("canvases", []):
            for image in canvas.get("images", []):
                resource = image.get("resource", {})
                img_url = resource.get("@id", "")
                if img_url.endswith(".jpg"):
                    jpg_urls.append(img_url)

    print(f"🖼️  {len(jpg_urls)} URLs extraites.")
    return jpg_urls

def save_to_txt(identifier, urls):
    filename = os.path.join(base_path, f"liste_{nom_dossier}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        for url in urls:
            f.write(url + "\n")
    return filename

In [ ]:
zip_name = f"images_{nom_dossier}.zip"

def zip_folder(folder_path, zip_name=zip_name):
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, _, files_in_dir in os.walk(folder_path):
            for file in files_in_dir:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, folder_path)
                z.write(full_path, arcname)

    print(f"Archive créée : {zip_name}")
    return zip_name

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

def download_images(txt_file):
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    with open(txt_file, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]

    progress = tqdm(urls, desc="📥 Téléchargement", unit="img")

    for i, url in enumerate(progress, start=1):
        filename = output_path / f"{identifier}_{i:04d}.jpg"

        # Mise à jour dynamique de la barre
        progress.set_postfix(page=i)

        if filename.exists():
            tqdm.write(f"⏭️ Page {i} déjà téléchargée")
            continue

        for attempt in range(1, max_retries + 1):
            try:
                url = url.replace("/full/full/", f"/full/{quality}/")
                response = requests.get(url, headers=headers, timeout=10)

                if response.status_code == 200:
                    filename.write_bytes(response.content)
                    tqdm.write(f"✅ Page {i} téléchargée")
                    break

                elif response.status_code == 429:
                    wait_time = delay * modif_delay
                    tqdm.write(f"⏳ 429 → tentative {attempt}/{max_retries}, pause {wait_time}s")
                    time.sleep(wait_time)

                elif response.status_code == 404:
                    tqdm.write("❌ Erreur 404 → arrêt du script")
                    sys.exit()

                else:
                    tqdm.write(f"⚠️ Erreur {response.status_code} (page {i})")

            except requests.exceptions.RequestException as e:
                tqdm.write(f"🌐 Erreur réseau (page {i}) : {e}")

        time.sleep(delay)

## Extraction des liens des images depuis le manifeste IIIF

In [ ]:
urls = extract_jpg_urls(identifier)
liste_images = save_to_txt(identifier, urls)

## Téléchargement des images

In [ ]:
for _ in range(2): # activer deux fois pour récupérer les images manquées
    download_images(liste_images)
    time.sleep(15)

## Téléchargement

In [ ]:
if drive == False and in_colab == True:
  from google.colab import files
  zip_file = zip_folder(output_folder)
  files.download(zip_file)